# RescueEye — Phase 3 Model Training

**University of Cebu – Banilad Campus · Capstone 2025**

This notebook trains two YOLO11 models on a free Colab T4 GPU:
1. **Victim Detection** — `yolo11s.pt` fine-tuned to detect persons in aerial/thermal disaster imagery
2. **Damage Classification** — `yolo11n-cls.pt` fine-tuned to classify flood / fire / structural / no-damage

**Estimated time:** ~30–60 min total on T4 GPU

**Before you start:**
- Runtime → Change runtime type → **T4 GPU**
- Mount your Google Drive (Step 1) to save trained weights
- Dataset downloads require internet access (already available in Colab)

## Step 1 — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Change these if you want a different save location
DRIVE_BASE = '/content/drive/MyDrive/RescueEye'
DATA_ROOT  = '/content/rescueeye_data'
MODELS_DIR = f'{DRIVE_BASE}/models'

os.makedirs(DATA_ROOT,  exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f'Data root  : {DATA_ROOT}')
print(f'Models dir : {MODELS_DIR}')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q ultralytics gdown Pillow numpy pyyaml scikit-learn matplotlib seaborn

import ultralytics
ultralytics.checks()
print('\n✓ All dependencies installed')

## Step 3 — Download Datasets

### Victim Detection — VisDrone (aerial pedestrian dataset)
VisDrone-DET2019 contains aerial images with person/pedestrian annotations —
closest publicly available proxy for SAR aerial imagery without account registration.

### Damage Classification — AIDER
AIDER (Aerial Image Dataset for Emergency Response) contains labelled aerial images
across flood, fire, collapsed building, and normal categories.

In [ ]:
import os, zipfile
from pathlib import Path

VIS_DIR = f'{DATA_ROOT}/raw/victim/visdrone'
os.makedirs(VIS_DIR, exist_ok=True)

# Download BOTH train and val splits for maximum data
splits_to_download = [
    ('train', 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip', '/content/visdrone_train.zip'),
    ('val',   'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',   '/content/visdrone_val.zip'),
]

for split_name, url, zip_path in splits_to_download:
    split_dir = f'{VIS_DIR}/{split_name}'
    if not os.path.exists(split_dir):
        print(f'Downloading VisDrone {split_name} split ...')
        !wget -q --show-progress -O {zip_path} {url}
        os.makedirs(split_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(split_dir)
        print(f'  Extracted to {split_dir}')
    else:
        print(f'VisDrone {split_name} already downloaded.')

for split_name, _, _ in splits_to_download:
    imgs = list(Path(f'{VIS_DIR}/{split_name}').rglob('*.jpg'))
    print(f'  {split_name}: {len(imgs)} images')

In [ ]:
import gdown

AIDER_DIR = f'{DATA_ROOT}/raw/damage/aider'

if not os.path.exists(AIDER_DIR):
    print('Downloading AIDER dataset ...')
    AIDER_GDRIVE_ID = '1XoFz5V6S9qjHn7UNdI8VBm0HUlsOQx2W'
    AIDER_ZIP = '/content/aider.zip'
    try:
        gdown.download(id=AIDER_GDRIVE_ID, output=AIDER_ZIP, quiet=False)
        os.makedirs(AIDER_DIR, exist_ok=True)
        with zipfile.ZipFile(AIDER_ZIP, 'r') as z:
            z.extractall(AIDER_DIR)
        print(f'Extracted to {AIDER_DIR}')
    except Exception as e:
        print(f'[WARN] AIDER download failed: {e}')
        print('Will use synthetic placeholder dataset instead.')
        AIDER_DIR = None
else:
    print('AIDER already downloaded.')

if AIDER_DIR:
    imgs = list(Path(AIDER_DIR).rglob('*.jpg')) + list(Path(AIDER_DIR).rglob('*.png'))
    print(f'  Images found: {len(imgs)}')

## Step 4 — Prepare Datasets (Convert & Split)

In [ ]:
# Convert VisDrone train + val splits to YOLOv8 format
import csv, shutil, yaml
from pathlib import Path
from PIL import Image

VICTIM_OUT  = Path(f'{DATA_ROOT}/victim')
PERSON_CATS = {1, 2}  # VisDrone: 1=pedestrian, 2=people

def visdrone_to_yolo(ann_path, img_w, img_h):
    lines = []
    with open(ann_path) as f:
        for row in csv.reader(f):
            if len(row) < 6: continue
            try: x,y,w,h,_,cat = int(row[0]),int(row[1]),int(row[2]),int(row[3]),row[4],int(row[5])
            except: continue
            if cat not in PERSON_CATS or w<=0 or h<=0: continue
            cx=(x+w/2)/img_w; cy=(y+h/2)/img_h
            lines.append(f'0 {cx:.6f} {cy:.6f} {w/img_w:.6f} {h/img_h:.6f}')
    return lines

# Use VisDrone's own train/val split (no manual shuffle needed)
for split in ('train', 'val'):
    split_raw = Path(f'{VIS_DIR}/{split}')
    # VisDrone zip extracts into a subdirectory — find images folder
    img_candidates = list(split_raw.rglob('images'))
    img_root = img_candidates[0] if img_candidates else split_raw
    imgs = sorted(img_root.glob('*.jpg'))

    img_out = VICTIM_OUT / 'images' / split
    lbl_out = VICTIM_OUT / 'labels' / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    count = 0
    for img_path in imgs:
        ann_candidates = list(split_raw.rglob('annotations'))
        ann_root = ann_candidates[0] if ann_candidates else split_raw
        ann_path = ann_root / img_path.with_suffix('.txt').name
        w, h = Image.open(img_path).size
        lines = visdrone_to_yolo(ann_path, w, h) if ann_path.exists() else []
        shutil.copy2(img_path, img_out / img_path.name)
        (lbl_out / img_path.with_suffix('.txt').name).write_text('\n'.join(lines))
        count += 1
    print(f'  {split}: {count} images')

# Use val split as test too
for folder in ('images', 'labels'):
    src = VICTIM_OUT / folder / 'val'
    dst = VICTIM_OUT / folder / 'test'
    if not dst.exists():
        shutil.copytree(src, dst)

victim_yaml = Path(f'{DATA_ROOT}/victim.yaml')
victim_yaml.write_text(yaml.dump({
    'path': str(VICTIM_OUT), 'train': 'images/train',
    'val': 'images/val', 'test': 'images/test', 'nc': 1, 'names': ['person']
}))
print(f'\n✓ victim.yaml → {victim_yaml}')
print(f'  Total images: {len(list((VICTIM_OUT/"images").rglob("*.jpg")))}')

In [ ]:
# Damage classification dataset
import random, shutil, yaml, numpy as np
from pathlib import Path
from PIL import Image, ImageDraw

DAMAGE_RAW = Path(f'{DATA_ROOT}/raw/damage')
DAMAGE_OUT = Path(f'{DATA_ROOT}/damage')
CLASSES    = ['flood_damage', 'fire_damage', 'structural_damage', 'no_damage']

def label_from_path(p):
    parts = [x.lower() for x in p.parts]
    for part in parts:
        if 'flood' in part: return 'flood_damage'
        if 'fire'  in part: return 'fire_damage'
        if 'collaps' in part or 'structural' in part or 'building' in part: return 'structural_damage'
        if 'normal' in part or 'no_damage' in part: return 'no_damage'
    return None

IMG_EXTS = {'.jpg', '.jpeg', '.png'}
buckets  = {c: [] for c in CLASSES}

for img in DAMAGE_RAW.rglob('*'):
    if img.suffix.lower() not in IMG_EXTS: continue
    lbl = label_from_path(img)
    if lbl: buckets[lbl].append(img)

total_real = sum(len(v) for v in buckets.values())

if total_real == 0:
    # No real data — generate synthetic placeholder dataset
    print('[INFO] No real damage images found — generating synthetic dataset ...')
    random.seed(1); np.random.seed(1)
    PALETTE = {
        'flood_damage':      (20,  80, 160),
        'fire_damage':       (200, 60,  10),
        'structural_damage': (120,100,  80),
        'no_damage':         (60, 120,  60),
    }
    COUNTS = {'train': 420, 'val': 120, 'test': 60}
    for split_name, per_class in COUNTS.items():
        for cls, base_color in PALETTE.items():
            out_dir = DAMAGE_OUT / split_name / cls
            out_dir.mkdir(parents=True, exist_ok=True)
            for i in range(per_class):
                noise = np.random.randint(-30, 30, (224, 224, 3), dtype=np.int16)
                arr = np.clip(np.array(base_color, dtype=np.int16) + noise, 0, 255).astype(np.uint8)
                img = Image.fromarray(arr)
                draw = ImageDraw.Draw(img)
                if cls == 'flood_damage':
                    for y in range(0, 224, 8):
                        draw.line([(0, y), (224, y + random.randint(-4,4))], fill=(30,100,200), width=1)
                elif cls == 'fire_damage':
                    for _ in range(20):
                        fx, fy = random.randint(0,212), random.randint(112,204)
                        draw.ellipse([fx, fy, fx+12, fy+20], fill=(255,140,0))
                elif cls == 'structural_damage':
                    for _ in range(10):
                        draw.line([(random.randint(0,224), random.randint(0,224)),
                                   (random.randint(0,224), random.randint(0,224))], fill=(80,60,50), width=2)
                img.save(out_dir / f'synth_{i:04d}.jpg', quality=85)
    print('[OK] Synthetic dataset generated: 600 images per class (420 train / 120 val / 60 test)')
else:
    print(f'Class distribution:')
    for cls, imgs in buckets.items():
        flag = '  ⚠ BELOW 500 MINIMUM' if len(imgs) < 500 else ''
        print(f'  {cls:<24} {len(imgs):>5}{flag}')
    random.seed(42)
    for cls, imgs in buckets.items():
        random.shuffle(imgs)
        n = len(imgs); n_tr = int(n*0.70); n_vl = int(n*0.20)
        for split, sl in [('train',imgs[:n_tr]),('val',imgs[n_tr:n_tr+n_vl]),('test',imgs[n_tr+n_vl:])]:
            out_dir = DAMAGE_OUT / split / cls
            out_dir.mkdir(parents=True, exist_ok=True)
            for src in sl: shutil.copy2(src, out_dir/src.name)

for split in ('train','val','test'):
    counts = {c: len(list((DAMAGE_OUT/split/c).glob('*'))) for c in CLASSES}
    print(f'  {split}: {sum(counts.values())} total  {counts}')

damage_yaml = Path(f'{DATA_ROOT}/damage.yaml')
damage_yaml.write_text(yaml.dump({
    'path': str(DAMAGE_OUT), 'train': 'train',
    'val': 'val', 'test': 'test', 'nc': 4, 'names': CLASSES
}))
print(f'\n✓ damage.yaml → {damage_yaml}')
print(f'✓ DAMAGE_OUT = {DAMAGE_OUT}')

## Step 5 — Train Victim Detection Model

Transfer learning from `yolo11s.pt` (YOLO11 small — better accuracy than YOLOv8s with fewer parameters) → fine-tuned on VisDrone aerial persons.

**Expected time:** ~25–40 min on T4 GPU

In [ ]:
from ultralytics import YOLO
import time, shutil, json
from pathlib import Path

VICTIM_RESULTS = f'{DATA_ROOT}/victim_results'
VICTIM_BEST    = f'{MODELS_DIR}/victim_best.pt'

model = YOLO('yolo11s.pt')
print('Training victim detection model (yolo11s, imgsz=1280) ...')
print('Expected time: ~60–90 min on T4 GPU\n')

t0 = time.time()
results = model.train(
    data=str(victim_yaml),
    epochs=100,
    imgsz=1280,
    batch=4,
    patience=15,
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    augment=True, mosaic=1.0, flipud=0.5, fliplr=0.5,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    copy_paste=0.3,
    project=VICTIM_RESULTS, name='victim_train', exist_ok=True,
    device=0, workers=2,
)
elapsed = time.time() - t0

map50   = float(results.results_dict.get('metrics/mAP50(B)', 0))
map5095 = float(results.results_dict.get('metrics/mAP50-95(B)', 0))
prec    = float(results.results_dict.get('metrics/precision(B)', 0))
rec     = float(results.results_dict.get('metrics/recall(B)', 0))

print(f'\n─── Victim Detection Results ───────────────────')
print(f'  mAP@0.5       : {map50:.4f}  (target >= 0.70)')
print(f'  mAP@0.5:0.95  : {map5095:.4f}')
print(f'  Precision     : {prec:.4f}')
print(f'  Recall        : {rec:.4f}')
print(f'  Training time : {elapsed/60:.1f} min')
print(f'  Status        : {"PASS ✓" if map50 >= 0.70 else "BELOW TARGET"}')

best_src = Path(VICTIM_RESULTS) / 'victim_train' / 'weights' / 'best.pt'
if best_src.exists():
    shutil.copy2(best_src, VICTIM_BEST)
    print(f'\n✓ Saved victim_best.pt -> {VICTIM_BEST}')

meta = {'map50': round(map50,4), 'map50_95': round(map5095,4),
        'precision': round(prec,4), 'recall': round(rec,4),
        'trained_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
Path(f'{MODELS_DIR}/victim_meta.json').write_text(json.dumps(meta, indent=2))
print('✓ Saved victim_meta.json')

## Step 6 — Train Damage Classification Model

Transfer learning from `yolo11n-cls.pt` → fine-tuned on AIDER (flood/fire/structural/no_damage).

**Expected time:** ~10–20 min on T4 GPU

In [ ]:
from ultralytics import YOLO
import time, shutil, json
from pathlib import Path

DAMAGE_RESULTS = f'{DATA_ROOT}/damage_results'
DAMAGE_BEST    = f'{MODELS_DIR}/damage_best.pt'

model = YOLO('yolo11n-cls.pt')  # YOLO11 nano classifier
print('Training damage classification model (yolo11n-cls) ...')

t0 = time.time()
results = model.train(
    data=str(DAMAGE_OUT),
    epochs=50, imgsz=224, batch=32, patience=10,
    optimizer='AdamW', lr0=0.001,
    augment=True, flipud=0.5, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    project=DAMAGE_RESULTS, name='damage_train', exist_ok=True,
    device=0, workers=2,
)
elapsed = time.time() - t0

top1 = float(results.results_dict.get('metrics/accuracy_top1', 0))
top5 = float(results.results_dict.get('metrics/accuracy_top5', 0))

print(f'\n─── Damage Classification Results ──────────────')
print(f'  Top-1 accuracy : {top1:.4f}  (target >= 0.75)')
print(f'  Top-5 accuracy : {top5:.4f}')
print(f'  Training time  : {elapsed/60:.1f} min')
print(f'  Status         : {"PASS" if top1 >= 0.75 else "BELOW TARGET"}')

best_src = Path(DAMAGE_RESULTS) / 'damage_train' / 'weights' / 'best.pt'
if best_src.exists():
    shutil.copy2(best_src, DAMAGE_BEST)
    print(f'\n✓ Saved damage_best.pt -> {DAMAGE_BEST}')

meta = {'accuracy_top1': round(top1,4), 'accuracy_top5': round(top5,4),
        'trained_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
Path(f'{MODELS_DIR}/damage_meta.json').write_text(json.dumps(meta, indent=2))
print('✓ Saved damage_meta.json')

## Step 7 — Evaluate Both Models

In [ ]:
from ultralytics import YOLO
import time, numpy as np
from pathlib import Path

print('=== Victim Detection — Test Split Evaluation ===')
v_model = YOLO(VICTIM_BEST)
v_val   = v_model.val(data=str(victim_yaml), split='test', verbose=False, conf=0.40,
                      project=f'{DATA_ROOT}/victim_results', name='final_eval', exist_ok=True)

map50   = float(v_val.results_dict.get('metrics/mAP50(B)', 0))
map5095 = float(v_val.results_dict.get('metrics/mAP50-95(B)', 0))
prec    = float(v_val.results_dict.get('metrics/precision(B)', 0))
rec     = float(v_val.results_dict.get('metrics/recall(B)', 0))
f1      = 2*prec*rec/(prec+rec+1e-8)

print(f'  mAP@0.5       {map50:.4f}')
print(f'  mAP@0.5:0.95  {map5095:.4f}')
print(f'  Precision     {prec:.4f}')
print(f'  Recall        {rec:.4f}')
print(f'  F1            {f1:.4f}')

test_imgs = list(Path(f'{DATA_ROOT}/victim/images/test').glob('*.jpg'))[:100]
times_v   = []
for img in test_imgs or [np.zeros((480,640,3),dtype='uint8')]*100:
    t0 = time.perf_counter()
    v_model(str(img) if isinstance(img, Path) else img, verbose=False)
    times_v.append((time.perf_counter()-t0)*1000)
print(f'  Avg inference  {sum(times_v)/len(times_v):.1f} ms')

In [ ]:
from ultralytics import YOLO
import time, numpy as np
from pathlib import Path

print('=== Damage Classification — Test Split Evaluation ===')
d_model = YOLO(DAMAGE_BEST)
d_val   = d_model.val(data=str(damage_yaml), split='test', verbose=False,
                      project=f'{DATA_ROOT}/damage_results', name='final_eval', exist_ok=True)

top1 = float(d_val.results_dict.get('metrics/accuracy_top1', 0))
top5 = float(d_val.results_dict.get('metrics/accuracy_top5', 0))
print(f'  Top-1 accuracy  {top1:.4f}')
print(f'  Top-5 accuracy  {top5:.4f}')

CLASSES = ['flood_damage','fire_damage','structural_damage','no_damage']
test_imgs = []
for cls in CLASSES:
    test_imgs += list(Path(f'{DATA_ROOT}/damage/test/{cls}').glob('*.jpg'))[:25]
times_d = []
for img in (test_imgs or [np.zeros((224,224,3),dtype='uint8')]*100):
    t0 = time.perf_counter()
    d_model(str(img) if isinstance(img, Path) else img, verbose=False)
    times_d.append((time.perf_counter()-t0)*1000)
print(f'  Avg inference   {sum(times_d)/len(times_d):.1f} ms')

## Step 8 — Latency Assertion (Objective 6)

In [ ]:
THRESHOLD_MS = 3000
avg_v    = sum(times_v) / len(times_v)
avg_d    = sum(times_d) / len(times_d)
combined = avg_v + avg_d

print(f'\n─── Latency Assertion ───────────────────────────')
print(f'  Victim detection avg   {avg_v:.1f} ms')
print(f'  Damage classification  {avg_d:.1f} ms')
print(f'  Combined               {combined:.1f} ms')
print(f'  Threshold              {THRESHOLD_MS} ms')

if combined < THRESHOLD_MS:
    print(f'\n  PASS — {combined:.0f}ms < {THRESHOLD_MS}ms')
else:
    print(f'\n  FAIL — {combined:.0f}ms >= {THRESHOLD_MS}ms')
    print('  Consider: reduce imgsz to 320 or export to ONNX')

## Step 9 — Final Summary

After this notebook completes:

1. Open your Google Drive → `RescueEye/models/` — you will find 4 files:
   - `victim_best.pt`
   - `damage_best.pt`
   - `victim_meta.json`
   - `damage_meta.json`

2. Download all 4 files and place them in `api/models/` on your local machine.

3. Restart FastAPI: `uvicorn main:app --port 8000`

4. Or hot-swap without restart: `curl -X POST http://localhost:8000/models/reload`

5. Verify the Dashboard shows **custom_v1** (green) model pills.

In [ ]:
print('='*55)
print('  RescueEye Phase 3 — Training Complete')
print('='*55)
print(f'  Victim mAP@0.5        : {map50:.4f}  {"PASS" if map50>=0.70 else "BELOW TARGET"}')
print(f'  Damage Top-1 accuracy : {top1:.4f}  {"PASS" if top1>=0.75 else "BELOW TARGET"}')
print(f'  Combined latency      : {combined:.0f}ms  {"PASS" if combined<3000 else "FAIL"}')
print()
print(f'  Weights saved to: {MODELS_DIR}')
print('  -> victim_best.pt')
print('  -> damage_best.pt')
print('  -> victim_meta.json')
print('  -> damage_meta.json')
print()
print('  Download from Google Drive -> place in api/models/ -> restart API')
print('='*55)

In [ ]:
from ultralytics import YOLO
import shutil
from pathlib import Path

!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MODELS_DIR  = '/content/drive/MyDrive/RescueEye/models'
VICTIM_BEST = f'{MODELS_DIR}/victim_best.pt'

print('Exporting victim model to ONNX ...')
model = YOLO(VICTIM_BEST)
export_path = model.export(format='onnx', imgsz=1280, simplify=True)

onnx_dest = f'{MODELS_DIR}/victim_best.onnx'
try:
    shutil.copy2(export_path, onnx_dest)
except shutil.SameFileError:
    pass  # already exported directly to Drive

print(f'✓ Saved victim_best.onnx -> {onnx_dest}')
print('Download this file and place it in api/models/victim_best.onnx')